<a href="https://colab.research.google.com/github/YogiReddy7396/Flipkart-Order-Issue-Resolution-Customer-Support-Analytics/blob/main/Flipkart_Order_Issue_Resolution_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Flipkart Order Issue Resolution & Customer Support Analytics



## Your Engineering Handover
You have joined the Order Support Operations team as a **Junior Python Developer**. The support lead spends hours switching between order records, finance refund logs, and customer complaints to understand which cases need attention. You have been asked to build a reproducible **order issue analysis workflow** in this notebook.

Your deliverable is **not an app or dashboard**. It is a trustworthy set of cleaned data tables, an order-level investigation report, and a concise evidence-based handover to Support Operations.

**How to work:** Make your own copy in Google Drive (`File → Save a copy in Drive`), upload the three provided files, complete the ten milestones in order, and run the assessment cell at the end. Write your own implementation in the code cells. Keep every required named output available for the next task and for assessment.

**Input files (only three):** `orders.csv` · `refunds.xlsx` · `customer_complaints.json`  
**Data:** Fictional training records; not actual Flipkart customer records.

> **Important:** The assessment at the end checks key output contracts and consistency. Passing it is not proof that every business calculation or written explanation is correct; reviewers may also inspect selected notebooks.

## Workspace setup · Data intake
Run the setup cell and upload the **three input files** when requested. Files stay in your current Colab session; if the runtime restarts, upload them again. Keep original filenames unchanged.

[Link to Dataset](https://drive.google.com/file/d/1F0o02Vzev9ar6OB3-TVWC7EGJcyiBKRv/view?usp=drive_link)

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('/content') if Path('/content').is_dir() else Path.cwd()
REQUIRED_FILES = ['orders.csv', 'refunds.xlsx', 'customer_complaints.json']

missing = [name for name in REQUIRED_FILES if not (DATA_DIR / name).exists()]
if missing:
    try:
        from google.colab import files
        print('Please upload:', ', '.join(missing))
        files.upload()
    except ImportError:
        print(f'Place these files in {DATA_DIR}: {missing}')

still_missing = [name for name in REQUIRED_FILES if not (DATA_DIR / name).exists()]
if still_missing:
    raise FileNotFoundError(f'Missing input files: {still_missing}')
print('All 3 source files are available.')

Please upload: orders.csv, refunds.xlsx, customer_complaints.json


Saving customer_complaints.json to customer_complaints.json
Saving orders.csv to orders.csv
Saving refunds.xlsx to refunds.xlsx
All 3 source files are available.


## Milestone 1 · Establish the order baseline
**From the team:** The operations lead has asked for an initial quality and volume briefing before accepting the three files into the reporting workflow. You are responsible for understanding what is in the Orders source, not interpreting business issues yet.

**Your responsibility:** Read `orders.csv` and inspect its structure. Report record counts, distinct order and customer counts, the column types, and missing values. Show a sample that helps a reviewer understand the data.

**Handover evidence:** Keep the raw DataFrame under **`orders_df`**. Show a clearly labeled baseline summary and missing-value review.

In [2]:
# Milestone 01: Establish the order baseline
# Implement your approach below and display the requested evidence.

In [3]:
import pandas as pd
orders_df = pd.read_csv("/content/orders.csv")
display(orders_df.head())

,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status
0,FKO000702,CUS00237,Bengaluru,2026-05-26,2026-05-30,NaN,Grocery,36082,Cancelled
1,FKO000636,CUS00689,Hyderabad,2026-04-19,2026-04-25,2026-04-24,Grocery,47326,Delivered
2,FKO001570,CUS00222,Pune,2026-01-23,2026-01-30,NaN,Home & Kitchen,43254,Shipped
3,FKO001303,CUS00035,Lucknow,2026-02-12,2026-02-17,2026-02-18,Home & Kitchen,22215,Delivered
4,FKO001208,CUS00619,Lucknow,2026-07-08,2026-07-11,2026-07-09,Home & Kitchen,2512,Delivered


In [4]:
print("Total rows & columns:", orders_df.shape)
print("Total rows:", orders_df.shape[0])
print("Total columns:", orders_df.shape[1])

Total rows & columns: (1624, 9)
Total rows: 1624
Total columns: 9


In [5]:
print("Unique orders:", orders_df["order_id"].nunique())
print("Unique customers:", orders_df["customer_id"].nunique())

Unique orders: 1600
Unique customers: 673


In [6]:
orders_df.dtypes

,0
order_id,object
customer_id,object
customer_city,object
order_date,object
promised_delivery_date,object
actual_delivery_date,object
product_category,object
order_amount,int64
order_status,object


In [7]:
orders_df.isnull().sum()

,0
order_id,0
customer_id,0
customer_city,0
order_date,0
promised_delivery_date,0
actual_delivery_date,492
product_category,0
order_amount,0
order_status,0


In [8]:
display(orders_df.head())

,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status
0,FKO000702,CUS00237,Bengaluru,2026-05-26,2026-05-30,NaN,Grocery,36082,Cancelled
1,FKO000636,CUS00689,Hyderabad,2026-04-19,2026-04-25,2026-04-24,Grocery,47326,Delivered
2,FKO001570,CUS00222,Pune,2026-01-23,2026-01-30,NaN,Home & Kitchen,43254,Shipped
3,FKO001303,CUS00035,Lucknow,2026-02-12,2026-02-17,2026-02-18,Home & Kitchen,22215,Delivered
4,FKO001208,CUS00619,Lucknow,2026-07-08,2026-07-11,2026-07-09,Home & Kitchen,2512,Delivered


In [9]:
# Clearly labeled missing-value review
missing_summary = pd.DataFrame({
    "Missing_Count": orders_df.isnull().sum(),
    "Missing_Percentage": (
        orders_df.isnull().mean() * 100
    ).round(2)
})

print("=== MISSING VALUE REVIEW ===")
display(missing_summary)

=== MISSING VALUE REVIEW ===


,Missing_Count,Missing_Percentage
order_id,0,0.0
customer_id,0,0.0
customer_city,0,0.0
order_date,0,0.0
promised_delivery_date,0,0.0
actual_delivery_date,492,30.3
product_category,0,0.0
order_amount,0,0.0
order_status,0,0.0


## Milestone 2 · Onboard finance and customer support data
**From the team:** The finance team exports refunds to a spreadsheet, while customer support maintains a JSON complaint log. The team wants assurance that both files can be used alongside the orders data.

**Your responsibility:** Load and inspect both sources, assess their shapes and fields, and describe the current distribution of refund outcomes and complaint types. Identify any data quality concerns worth revisiting during cleaning.

**Handover evidence:** Keep raw DataFrames as **`refunds_df`** and **`complaints_df`**. Present counts and readable category/status summaries.

In [10]:
# Milestone 02: Onboard finance and customer support data
# Implement your approach below and display the requested evidence.

In [11]:
# Load the refund and complaint datasets
refunds_df = pd.read_excel("/content/refunds.xlsx")
complaints_df = pd.read_json("/content/customer_complaints.json")

In [12]:
print("REFUND DATASET")
print("Rows and columns:", refunds_df.shape)
print("Column names:", refunds_df.columns.tolist())

print()

print("COMPLAINT DATASET")
print("Rows and columns:", complaints_df.shape)
print("Column names:", complaints_df.columns.tolist())

REFUND DATASET
Rows and columns: (1371, 5)
Column names: ['refund_id', 'order_id', 'refund_amount', 'refund_date', 'refund_status']

COMPLAINT DATASET
Rows and columns: (1525, 6)
Column names: ['complaint_id', 'order_id', 'customer_id', 'complaint_date', 'complaint_type', 'complaint_status']


In [13]:
print("REFUND DATA TYPES")
refunds_df.info()

print("\nCOMPLAINT DATA TYPES")
complaints_df.info()

REFUND DATA TYPES
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1371 entries, 0 to 1370
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   refund_id      1371 non-null   object
 1   order_id       1371 non-null   object
 2   refund_amount  1371 non-null   int64 
 3   refund_date    1371 non-null   object
 4   refund_status  1371 non-null   object
dtypes: int64(1), object(4)
memory usage: 53.7+ KB

COMPLAINT DATA TYPES
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1525 entries, 0 to 1524
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   complaint_id      1525 non-null   object
 1   order_id          1525 non-null   object
 2   customer_id       1525 non-null   object
 3   complaint_date    1525 non-null   object
 4   complaint_type    1525 non-null   object
 5   complaint_status  1525 non-null   object
dtypes: object(6)
memory usa

In [14]:
print("SAMPLE REFUND RECORDS")
display(refunds_df.head())

print("\nSAMPLE COMPLAINT RECORDS")
display(complaints_df.head())

SAMPLE REFUND RECORDS


,refund_id,order_id,refund_amount,refund_date,refund_status
0,RFD000291,FKO000086,15535,2026-06-16,Completed
1,RFD000103,FKO000927,4051,2026-07-01,Completed
2,RFD000518,FKO000907,8761,2026-02-24,Completed
3,RFD000217,FKO000233,16436,2026-04-12,Failed
4,RFD000800,FKO000474,2047,2026-05-02,Completed



SAMPLE COMPLAINT RECORDS


,complaint_id,order_id,customer_id,complaint_date,complaint_type,complaint_status
0,CMP001122,FKO001270,CUS00287,2026-02-08,Product Quality,Resolved
1,CMP000032,FKO001072,CUS00559,2026-03-20,Delivery Delay,resolved
2,CMP000582,FKO000243,CUS00684,2026-08-28,Cancellation Issue,Open
3,CMP000622,FKO001525,CUS00646,2026-08-28,Delivery Delay,Open
4,CMP000846,FKO000828,CUS00149,2026-03-22,Product Quality,In Progress


In [15]:
print("REFUND STATUS COUNTS")

display(
    refunds_df["refund_status"]
    .fillna("Missing")
    .astype(str)
    .str.strip()
    .value_counts()
    .rename_axis("Refund_Status")
    .reset_index(name="Count")
)

REFUND STATUS COUNTS


,Refund_Status,Count
0,Completed,662
1,Pending,429
2,Failed,174
3,completed,56
4,pending,30
5,failed,20


In [16]:
print("COMPLAINT TYPES")

display(
    complaints_df["complaint_type"]
    .fillna("Missing")
    .astype(str)
    .str.strip()
    .value_counts()
    .rename_axis("Complaint_Type")
    .reset_index(name="Count")
)

print()

print("COMPLAINT STATUSES")

display(
    complaints_df["complaint_status"]
    .fillna("Missing")
    .astype(str)
    .str.strip()
    .value_counts()
    .rename_axis("Complaint_Status")
    .reset_index(name="Count")
)

COMPLAINT TYPES


,Complaint_Type,Count
0,Delivery Delay,240
1,Product Quality,234
2,Missing Item,209
3,Cancellation Issue,203
4,Damaged Product,200
5,Wrong Product,192
6,Refund Issue,190
7,product quality,12
8,missing item,10
9,wrong product,8



COMPLAINT STATUSES


,Complaint_Status,Count
0,Resolved,584
1,Open,488
2,In Progress,357
3,resolved,43
4,in progress,27
5,open,26


In [17]:
print("REFUND MISSING VALUES")
display(refunds_df.isnull().sum())

print("\nCOMPLAINT MISSING VALUES")
display(complaints_df.isnull().sum())

print("Duplicate refund IDs:",
      refunds_df["refund_id"].duplicated().sum())

print("Duplicate complaint IDs:",
      complaints_df["complaint_id"].duplicated().sum())

REFUND MISSING VALUES


,0
refund_id,0
order_id,0
refund_amount,0
refund_date,0
refund_status,0



COMPLAINT MISSING VALUES


,0
complaint_id,0
order_id,0
customer_id,0
complaint_date,0
complaint_type,0
complaint_status,0


Duplicate refund IDs: 21
Duplicate complaint IDs: 25


## Milestone 3 · Prepare reliable order records
**From the team:** Operations has noticed repeated IDs, inconsistent order labels, and delivery-date issues. Without cleaning, the team may miscount orders or report misleading delays.

**Your responsibility:** Produce a clean order-level source. Keep one record per order ID, normalize order statuses, ensure order amounts are numeric, and make order and delivery dates usable for comparisons. Treat non-delivered orders with no actual delivery date differently from invalid/missing dates on delivered orders. Explain your handling of ambiguous values.

**Handover evidence:** Create **`orders_clean`**, preserving key source fields: `order_id`, `customer_id`, `customer_city`, `order_date`, `promised_delivery_date`, `actual_delivery_date`, `product_category`, `order_amount`, `order_status`. Include a concise cleaning note.

In [18]:
# Milestone 03: Prepare reliable order records
# Implement your approach below and display the requested evidence.

In [19]:
# Checking duplicate order IDs
print("Total records:", len(orders_df))
print("Unique order IDs:", orders_df["order_id"].nunique())
print("Duplicate order ID records:",orders_df["order_id"].duplicated().sum())

Total records: 1624
Unique order IDs: 1600
Duplicate order ID records: 24


In [20]:
# Display all records with repeated order IDs
duplicate_orders = orders_df[
    orders_df["order_id"].duplicated(keep=False)
].sort_values("order_id")

display(duplicate_orders)

,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status
776,FKO000114,CUS00012,Ahmedabad,2026-06-18,2026-06-21,2026-06-23,Fashion,40806,delivered
953,FKO000114,CUS00012,Ahmedabad,2026-06-18,2026-06-21,2026-06-23,Fashion,40806,delivered
757,FKO000120,CUS00581,Bengaluru,2026-03-08,2026-03-16,NaN,Fashion,36792,Shipped
401,FKO000120,CUS00581,Bengaluru,2026-03-08,2026-03-16,NaN,Fashion,36792,Shipped
1551,FKO000173,CUS00639,Kolkata,2026-04-08,2026-04-15,2026-04-18,Grocery,8650,Delivered
448,FKO000173,CUS00639,Kolkata,2026-04-08,2026-04-15,2026-04-18,Grocery,8650,Delivered
543,FKO000186,CUS00227,Hyderabad,2026-06-06,2026-06-12,2026-06-13,Sports,3910,Delivered
116,FKO000186,CUS00227,Hyderabad,2026-06-06,2026-06-12,2026-06-13,Sports,3910,Delivered
862,FKO000200,CUS00393,Bengaluru,2026-06-06,2026-06-12,2026-06-14,Home & Kitchen,40980,Delivered
265,FKO000200,CUS00393,Bengaluru,2026-06-06,2026-06-12,2026-06-14,Home & Kitchen,40980,Delivered


In [21]:
# Creating  a copy for cleaning
orders_clean = orders_df.copy()
print("Original dataset shape:", orders_df.shape)
print("Working dataset shape:", orders_clean.shape)

display(orders_clean.head())

Original dataset shape: (1624, 9)
Working dataset shape: (1624, 9)


,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status
0,FKO000702,CUS00237,Bengaluru,2026-05-26,2026-05-30,NaN,Grocery,36082,Cancelled
1,FKO000636,CUS00689,Hyderabad,2026-04-19,2026-04-25,2026-04-24,Grocery,47326,Delivered
2,FKO001570,CUS00222,Pune,2026-01-23,2026-01-30,NaN,Home & Kitchen,43254,Shipped
3,FKO001303,CUS00035,Lucknow,2026-02-12,2026-02-17,2026-02-18,Home & Kitchen,22215,Delivered
4,FKO001208,CUS00619,Lucknow,2026-07-08,2026-07-11,2026-07-09,Home & Kitchen,2512,Delivered


In [22]:
#Exact duplicate rows
print("Exact duplicate rows:", orders_clean.duplicated().sum())

Exact duplicate rows: 24


In [23]:
# Before Cleaning
print("Before cleaning:", orders_clean.columns.tolist())

orders_clean.columns = (orders_clean.columns.str.strip().str.lower().str.replace(" ", "_"))
#After Cleaning
print("After cleaning:", orders_clean.columns.tolist())

Before cleaning: ['order_id', 'customer_id', 'customer_city', 'order_date', 'promised_delivery_date', 'actual_delivery_date', 'product_category', 'order_amount', 'order_status']
After cleaning: ['order_id', 'customer_id', 'customer_city', 'order_date', 'promised_delivery_date', 'actual_delivery_date', 'product_category', 'order_amount', 'order_status']


In [24]:
# Identify text columns
text_columns = orders_clean.select_dtypes(include="object").columns

for column in text_columns:
    orders_clean[column] = orders_clean[column].str.strip()

# Standardize order statuses
orders_clean["order_status"] = (
    orders_clean["order_status"].str.title()
)

# Check the results
print("Order status counts:")
print(orders_clean["order_status"].value_counts(dropna=False))
# After cleaning , Checking the total rows are matching or not --> (Matched)
orders_clean.shape[0]

Order status counts:
order_status
Delivered    1145
Shipped       216
Pending       164
Cancelled      99
Name: count, dtype: int64


1624

In [25]:
orders_clean.dtypes

,0
order_id,object
customer_id,object
customer_city,object
order_date,object
promised_delivery_date,object
actual_delivery_date,object
product_category,object
order_amount,int64
order_status,object


In [26]:
# Data type before conversion
print("Before conversion:", orders_clean["order_amount"].dtype)

# Convert order_amount to numeric
orders_clean["order_amount"] = pd.to_numeric(orders_clean["order_amount"],errors="coerce")

# Data type after conversion
print("After conversion:", orders_clean["order_amount"].dtype)

# Count missing values after conversion
print("Missing or invalid order amounts:",orders_clean["order_amount"].isna().sum())

Before conversion: int64
After conversion: int64
Missing or invalid order amounts: 0


In [27]:
date_columns = [
    "order_date",
    "promised_delivery_date",
    "actual_delivery_date"
]

# Convert each column to datetime
for column in date_columns:
    orders_clean[column] = pd.to_datetime(
        orders_clean[column],
        errors="coerce"
    )

# Checking data types
print("Date column data types:")
print(orders_clean[date_columns].dtypes)

# Checking missing values
print("\nMissing dates after conversion:")
print(orders_clean[date_columns].isna().sum())

Date column data types:
order_date                datetime64[ns]
promised_delivery_date    datetime64[ns]
actual_delivery_date      datetime64[ns]
dtype: object

Missing dates after conversion:
order_date                  0
promised_delivery_date      0
actual_delivery_date      492
dtype: int64


In [28]:
# Now checking datatypes
orders_clean.dtypes

,0
order_id,object
customer_id,object
customer_city,object
order_date,datetime64[ns]
promised_delivery_date,datetime64[ns]
actual_delivery_date,datetime64[ns]
product_category,object
order_amount,int64
order_status,object


In [29]:
#Reverify with dataset
orders_clean.isnull().sum()

,0
order_id,0
customer_id,0
customer_city,0
order_date,0
promised_delivery_date,0
actual_delivery_date,492
product_category,0
order_amount,0
order_status,0


In [30]:
# Check order statuses for orders without an actual delivery date

missing_delivery_orders = orders_clean[orders_clean["actual_delivery_date"].isna()]

print("Orders missing actual delivery dates:",len(missing_delivery_orders))

print("\nOrder status breakdown:")
print(missing_delivery_orders["order_status"].value_counts(dropna=False))

Orders missing actual delivery dates: 492

Order status breakdown:
order_status
Shipped      216
Pending      164
Cancelled     99
Delivered     13
Name: count, dtype: int64


In [31]:
# Checking missing values in every column

missing_summary = pd.DataFrame({
    "Missing_Count": orders_clean.isna().sum(),
    "Missing_Percentage": (orders_clean.isna().sum() / len(orders_clean) * 100).round(2)})

display(missing_summary)

,Missing_Count,Missing_Percentage
order_id,0,0.0
customer_id,0,0.0
customer_city,0,0.0
order_date,0,0.0
promised_delivery_date,0,0.0
actual_delivery_date,492,30.3
product_category,0,0.0
order_amount,0,0.0
order_status,0,0.0


In [32]:
# Check duplicate order IDs after cleaning

print("Total rows:", len(orders_clean))
print("Unique order IDs:", orders_clean["order_id"].nunique())
print("Repeated order ID rows:", orders_clean["order_id"].duplicated().sum())

# Inspect all rows with repeated order IDs
duplicate_orders = orders_clean[orders_clean["order_id"].duplicated(keep=False)].sort_values("order_id")

display(duplicate_orders)

Total rows: 1624
Unique order IDs: 1600
Repeated order ID rows: 24


,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status
776,FKO000114,CUS00012,Ahmedabad,2026-06-18,2026-06-21,2026-06-23,Fashion,40806,Delivered
953,FKO000114,CUS00012,Ahmedabad,2026-06-18,2026-06-21,2026-06-23,Fashion,40806,Delivered
757,FKO000120,CUS00581,Bengaluru,2026-03-08,2026-03-16,NaT,Fashion,36792,Shipped
401,FKO000120,CUS00581,Bengaluru,2026-03-08,2026-03-16,NaT,Fashion,36792,Shipped
1551,FKO000173,CUS00639,Kolkata,2026-04-08,2026-04-15,2026-04-18,Grocery,8650,Delivered
448,FKO000173,CUS00639,Kolkata,2026-04-08,2026-04-15,2026-04-18,Grocery,8650,Delivered
543,FKO000186,CUS00227,Hyderabad,2026-06-06,2026-06-12,2026-06-13,Sports,3910,Delivered
116,FKO000186,CUS00227,Hyderabad,2026-06-06,2026-06-12,2026-06-13,Sports,3910,Delivered
862,FKO000200,CUS00393,Bengaluru,2026-06-06,2026-06-12,2026-06-14,Home & Kitchen,40980,Delivered
265,FKO000200,CUS00393,Bengaluru,2026-06-06,2026-06-12,2026-06-14,Home & Kitchen,40980,Delivered


In [33]:
# Remove exact duplicate rows
orders_clean = orders_clean.drop_duplicates().copy()

# Verify the result
print("Rows after removing exact duplicates:", len(orders_clean))
print("Unique order IDs:", orders_clean["order_id"].nunique())
print("Repeated order ID rows:",orders_clean["order_id"].duplicated().sum())

Rows after removing exact duplicates: 1600
Unique order IDs: 1600
Repeated order ID rows: 0


In [34]:
required_columns = [
    "order_id",
    "customer_id",
    "customer_city",
    "order_date",
    "promised_delivery_date",
    "actual_delivery_date",
    "product_category",
    "order_amount",
    "order_status"
]

orders_clean = orders_clean[required_columns]

# Verify the dataset
print("Dataset shape:", orders_clean.shape)
print("Column names:", orders_clean.columns.tolist())

display(orders_clean.head())

Dataset shape: (1600, 9)
Column names: ['order_id', 'customer_id', 'customer_city', 'order_date', 'promised_delivery_date', 'actual_delivery_date', 'product_category', 'order_amount', 'order_status']


,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status
0,FKO000702,CUS00237,Bengaluru,2026-05-26,2026-05-30,NaT,Grocery,36082,Cancelled
1,FKO000636,CUS00689,Hyderabad,2026-04-19,2026-04-25,2026-04-24,Grocery,47326,Delivered
2,FKO001570,CUS00222,Pune,2026-01-23,2026-01-30,NaT,Home & Kitchen,43254,Shipped
3,FKO001303,CUS00035,Lucknow,2026-02-12,2026-02-17,2026-02-18,Home & Kitchen,22215,Delivered
4,FKO001208,CUS00619,Lucknow,2026-07-08,2026-07-11,2026-07-09,Home & Kitchen,2512,Delivered


In [35]:
# Final quality checks

print("Missing values:")
print(orders_clean.isna().sum())

print("\nDuplicate order IDs:",orders_clean["order_id"].duplicated().sum())

print("\nData types:")
print(orders_clean.dtypes)

print("\nUnique order statuses:")
print(orders_clean["order_status"].value_counts(dropna=False))

Missing values:
order_id                    0
customer_id                 0
customer_city               0
order_date                  0
promised_delivery_date      0
actual_delivery_date      487
product_category            0
order_amount                0
order_status                0
dtype: int64

Duplicate order IDs: 0

Data types:
order_id                          object
customer_id                       object
customer_city                     object
order_date                datetime64[ns]
promised_delivery_date    datetime64[ns]
actual_delivery_date      datetime64[ns]
product_category                  object
order_amount                       int64
order_status                      object
dtype: object

Unique order statuses:
order_status
Delivered    1126
Shipped       213
Pending       162
Cancelled      99
Name: count, dtype: int64


In [36]:
print("Total rows:", len(orders_clean))
print("Unique order IDs:", orders_clean["order_id"].nunique())
print("Duplicate order IDs:",orders_clean["order_id"].duplicated().sum())

Total rows: 1600
Unique order IDs: 1600
Duplicate order IDs: 0


### Milestone 3 — Cleaning Note

The original dataset was preserved in `orders_df`, and a separate DataFrame, `orders_clean`, was created for cleaning. Column names were standardized to lowercase with underscores, text values were stripped of leading and trailing spaces, and order statuses were normalized. The order amount was converted to a numeric data type, and order and delivery dates were converted to datetime format.

Exact duplicate rows were removed, leaving 1,600 unique order IDs. The 487 missing actual delivery dates were retained rather than replaced with guessed values. These records were reviewed by order status to identify potentially undelivered orders and delivered orders requiring further investigation.

## Milestone 4 · Make issue records consistent
**From the team:** Refunds and customer complaints were compiled by different teams. A single order can have multiple genuine issue records, so removing all repeated order IDs would lose useful evidence.

**Your responsibility:** Create dependable refund and complaint data. Resolve duplicate refund/complaint IDs, standardize status labels, make refund amounts and all event dates usable, and surface records with missing order links. Preserve legitimate multiple issues against the same order.

**Handover evidence:** Create **`refunds_clean`** and **`complaints_clean`**, retaining the original business fields and consistent status values. Show a small before/after quality summary.

In [37]:
# Milestone 04: Make issue records consistent
# Implement your approach below and display the requested evidence.

In [38]:
print("Refund records:", len(refunds_df))

print("\nColumn names:")
print(refunds_df.columns.tolist())

print("\nData types:")
print(refunds_df.dtypes)

print("\nFirst five records:")
display(refunds_df.head())

Refund records: 1371

Column names:
['refund_id', 'order_id', 'refund_amount', 'refund_date', 'refund_status']

Data types:
refund_id        object
order_id         object
refund_amount     int64
refund_date      object
refund_status    object
dtype: object

First five records:


,refund_id,order_id,refund_amount,refund_date,refund_status
0,RFD000291,FKO000086,15535,2026-06-16,Completed
1,RFD000103,FKO000927,4051,2026-07-01,Completed
2,RFD000518,FKO000907,8761,2026-02-24,Completed
3,RFD000217,FKO000233,16436,2026-04-12,Failed
4,RFD000800,FKO000474,2047,2026-05-02,Completed


In [39]:
# Create a working copy and check duplicates

refunds_clean = refunds_df.copy()

print("Original refund records:", len(refunds_df))
print("Working refund records:", len(refunds_clean))

print("\nExact duplicate rows:",refunds_clean.duplicated().sum())

print("\nDuplicate refund IDs:",refunds_clean["refund_id"].duplicated().sum())

print("\nRecords with duplicate refund IDs:")
duplicate_refunds = refunds_clean[refunds_clean["refund_id"].duplicated(keep=False)].sort_values("refund_id")

display(duplicate_refunds)

Original refund records: 1371
Working refund records: 1371

Exact duplicate rows: 21

Duplicate refund IDs: 21

Records with duplicate refund IDs:


,refund_id,order_id,refund_amount,refund_date,refund_status
1270,RFD000020,FKO000621,1898,2026-08-18,Completed
399,RFD000020,FKO000621,1898,2026-08-18,Completed
736,RFD000266,FKO000954,10719,2026-05-18,Pending
585,RFD000266,FKO000954,10719,2026-05-18,Pending
844,RFD000290,FKO000549,451,2026-03-18,Completed
461,RFD000290,FKO000549,451,2026-03-18,Completed
1304,RFD000352,FKO001020,11508,2026-06-07,Pending
584,RFD000352,FKO001020,11508,2026-06-07,Pending
934,RFD000358,FKO000622,1920,2026-04-10,Failed
359,RFD000358,FKO000622,1920,2026-04-10,Failed


In [40]:
# Check refund status consistency

print("Refund status counts before cleaning:")
print(refunds_clean["refund_status"].value_counts(dropna=False))

print("\nUnique refund statuses:")
print(refunds_clean["refund_status"].unique())

Refund status counts before cleaning:
refund_status
Completed      662
Pending        429
Failed         174
 completed      56
 pending        30
 failed         20
Name: count, dtype: int64

Unique refund statuses:
['Completed' 'Failed' 'Pending' ' completed ' ' pending ' ' failed ']


In [41]:
# Standardize refund statuses

refunds_clean["refund_status"] = (refunds_clean["refund_status"].astype("string").str.strip().str.title())

print("Refund status counts after cleaning:")
print(refunds_clean["refund_status"].value_counts(dropna=False))

Refund status counts after cleaning:
refund_status
Completed    718
Pending      459
Failed       194
Name: count, dtype: Int64


In [42]:
# Convert refund amounts and dates

print("Before conversion:")
print("Refund amount dtype:", refunds_clean["refund_amount"].dtype)
print("Refund date dtype:", refunds_clean["refund_date"].dtype)

# Convert refund amounts to numeric
refunds_clean["refund_amount"] = pd.to_numeric(refunds_clean["refund_amount"], errors="coerce")

# Convert refund dates to datetime
refunds_clean["refund_date"] = pd.to_datetime(refunds_clean["refund_date"], errors="coerce")

print("\nAfter conversion:")
print("Refund amount dtype:", refunds_clean["refund_amount"].dtype)
print("Refund date dtype:", refunds_clean["refund_date"].dtype)

print("\nMissing or invalid refund amounts:",
      refunds_clean["refund_amount"].isna().sum())

print("Missing or invalid refund dates:",
      refunds_clean["refund_date"].isna().sum())

Before conversion:
Refund amount dtype: int64
Refund date dtype: object

After conversion:
Refund amount dtype: int64
Refund date dtype: datetime64[ns]

Missing or invalid refund amounts: 0
Missing or invalid refund dates: 0


In [43]:
print("Missing order IDs:")
print(refunds_clean["order_id"].isnull().sum())

Missing order IDs:
0


In [44]:
print("Unmatched order IDs:")
print((~refunds_clean["order_id"].isin(orders_clean["order_id"])).sum())

Unmatched order IDs:
18


In [45]:
# Check duplicate refund IDs

print("Duplicate refund IDs:")
print(refunds_clean["refund_id"].duplicated().sum())

Duplicate refund IDs:
21


In [46]:
# Check missing values in refunds_clean
print(refunds_clean.isnull().sum())

refund_id        0
order_id         0
refund_amount    0
refund_date      0
refund_status    0
dtype: int64


In [47]:
print("Total refund records:", len(refunds_clean))
print("Unique refund IDs:", refunds_clean["refund_id"].nunique())
print("Duplicate refund IDs:",refunds_clean["refund_id"].duplicated().sum())

Total refund records: 1371
Unique refund IDs: 1350
Duplicate refund IDs: 21


In [48]:
# Inspect customer complaints
print("Total complaint records:", len(complaints_df))

print("\nColumn names:")
print(complaints_df.columns.tolist())

print("\nData Types:")
print(complaints_df.dtypes)

print("\nFirst five records:")
display(complaints_df.head())

Total complaint records: 1525

Column names:
['complaint_id', 'order_id', 'customer_id', 'complaint_date', 'complaint_type', 'complaint_status']

Data Types:
complaint_id        object
order_id            object
customer_id         object
complaint_date      object
complaint_type      object
complaint_status    object
dtype: object

First five records:


,complaint_id,order_id,customer_id,complaint_date,complaint_type,complaint_status
0,CMP001122,FKO001270,CUS00287,2026-02-08,Product Quality,Resolved
1,CMP000032,FKO001072,CUS00559,2026-03-20,Delivery Delay,resolved
2,CMP000582,FKO000243,CUS00684,2026-08-28,Cancellation Issue,Open
3,CMP000622,FKO001525,CUS00646,2026-08-28,Delivery Delay,Open
4,CMP000846,FKO000828,CUS00149,2026-03-22,Product Quality,In Progress


In [49]:
# Create a working copy of complaints

complaints_clean = complaints_df.copy()

print("Original complaint records:", len(complaints_df))
print("Working complaint records:", len(complaints_clean))

print("\nDuplicate complaint IDs:")
print(complaints_clean["complaint_id"].duplicated().sum())

Original complaint records: 1525
Working complaint records: 1525

Duplicate complaint IDs:
25


In [50]:
print(complaints_clean["complaint_status"].value_counts(dropna=False))

complaint_status
Resolved         584
Open             488
In Progress      357
 resolved         43
 in progress      27
 open             26
Name: count, dtype: int64


In [51]:
# Standardize complaint status names

complaints_clean["complaint_status"] = (complaints_clean["complaint_status"].str.strip().str.title())
print(complaints_clean["complaint_status"].value_counts(dropna=False))

complaint_status
Resolved       627
Open           514
In Progress    384
Name: count, dtype: int64


In [52]:
# Check duplicate complaint IDs

print("Total complaint records:", len(complaints_clean))

print("Unique complaint IDs:",complaints_clean["complaint_id"].nunique())

print("Duplicate complaint IDs:",complaints_clean["complaint_id"].duplicated().sum())

Total complaint records: 1525
Unique complaint IDs: 1500
Duplicate complaint IDs: 25


In [53]:
# Convert complaint date to datetime
complaints_clean["complaint_date"] = pd.to_datetime(complaints_clean["complaint_date"], errors="coerce")

print("Complaint date data type:",complaints_clean["complaint_date"].dtype)

print("Missing complaint dates:",complaints_clean["complaint_date"].isnull().sum())

Complaint date data type: datetime64[ns]
Missing complaint dates: 0


In [54]:
# Check missing values in complaints_clean

print(complaints_clean.isnull().sum())

complaint_id        0
order_id            0
customer_id         0
complaint_date      0
complaint_type      0
complaint_status    0
dtype: int64


In [55]:
# Check unmatched complaint orders

unmatched_complaints = complaints_clean[~complaints_clean["order_id"].isin(orders_clean["order_id"])]
print("Unmatched complaint records:", len(unmatched_complaints))

Unmatched complaint records: 21


In [56]:
# Data quality summary

print("REFUNDS")
print("Before cleaning:", len(refunds_df))
print("After cleaning:", len(refunds_clean))
print("Missing values after cleaning:",
      refunds_clean.isnull().sum().sum())

print("\nCOMPLAINTS")
print("Before cleaning:", len(complaints_df))
print("After cleaning:", len(complaints_clean))
print("Missing values after cleaning:",
      complaints_clean.isnull().sum().sum())

REFUNDS
Before cleaning: 1371
After cleaning: 1371
Missing values after cleaning: 0

COMPLAINTS
Before cleaning: 1525
After cleaning: 1525
Missing values after cleaning: 0


In [57]:
# Before-and-after data quality summary

print("REFUNDS")
print("Records before:", len(refunds_df))
print("Records after:", len(refunds_clean))
print("Duplicate refund IDs before:",refunds_df["refund_id"].duplicated().sum())
print("Duplicate refund IDs after:",refunds_clean["refund_id"].duplicated().sum())

print("\nCOMPLAINTS")
print("Records before:", len(complaints_df))
print("Records after:", len(complaints_clean))
print("Duplicate complaint IDs before:",complaints_df["complaint_id"].duplicated().sum())
print("Duplicate complaint IDs after:",complaints_clean["complaint_id"].duplicated().sum())

REFUNDS
Records before: 1371
Records after: 1371
Duplicate refund IDs before: 21
Duplicate refund IDs after: 21

COMPLAINTS
Records before: 1525
Records after: 1525
Duplicate complaint IDs before: 25
Duplicate complaint IDs after: 25


**Milestone 4: Issue Data Cleaning Summary**

- Created separate working copies named `refunds_clean` and `complaints_clean`.
- Standardized refund and complaint status capitalization and removed extra spaces from refund statuses.
- Converted refund amounts to numeric values and refund dates to datetime format. Complaint dates were also converted to datetime format.
- Checked duplicate refund IDs and complaint IDs without automatically deleting records.
- Preserved multiple refund and complaint records linked to the same order.
- Found 18 refund records and 21 complaint records with unmatched order IDs. These records were retained for investigation.
- Checked missing values in the cleaned datasets.

## Milestone 5 · Measure delivery reliability
**From the team:** The delivery lead needs a factual picture of late deliveries and where they occur before assigning support capacity.

**Your responsibility:** Analyze order volumes by status and revenue by product category. For completed deliveries with valid promised and actual dates, measure lateness relative to the promised day. Identify delayed orders and compare delayed-order counts across cities. Do not classify an undelivered shipment as an on-time delivery simply because its actual date is blank.

**Handover evidence:** Add numeric **`delivery_delay_days`** to **`orders_clean`** (positive = days late, zero = on date, negative = early; missing where not computable). Create **`city_delay_summary`** containing `customer_city` and `delayed_order_count`. Display the top five cities.

In [58]:
# Milestone 05: Measure delivery reliability
# Implement your approach below and display the requested evidence.

In [59]:
orders_clean.head()

,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status
0,FKO000702,CUS00237,Bengaluru,2026-05-26,2026-05-30,NaT,Grocery,36082,Cancelled
1,FKO000636,CUS00689,Hyderabad,2026-04-19,2026-04-25,2026-04-24,Grocery,47326,Delivered
2,FKO001570,CUS00222,Pune,2026-01-23,2026-01-30,NaT,Home & Kitchen,43254,Shipped
3,FKO001303,CUS00035,Lucknow,2026-02-12,2026-02-17,2026-02-18,Home & Kitchen,22215,Delivered
4,FKO001208,CUS00619,Lucknow,2026-07-08,2026-07-11,2026-07-09,Home & Kitchen,2512,Delivered


In [60]:
# Now calculate the delivery delay days safely
orders_clean["delivery_delay_days"] = (orders_clean["actual_delivery_date"] - orders_clean["promised_delivery_date"]).dt.days

# Display the preview
display(orders_clean[["order_id", "promised_delivery_date", "actual_delivery_date", "delivery_delay_days"]].head(10))

,order_id,promised_delivery_date,actual_delivery_date,delivery_delay_days
0,FKO000702,2026-05-30,NaT,NaN
1,FKO000636,2026-04-25,2026-04-24,-1.0
2,FKO001570,2026-01-30,NaT,NaN
3,FKO001303,2026-02-17,2026-02-18,1.0
4,FKO001208,2026-07-11,2026-07-09,-2.0
5,FKO001595,2026-05-24,2026-05-28,4.0
6,FKO000849,2026-03-13,NaT,NaN
7,FKO001348,2026-07-11,NaT,NaN
8,FKO001241,2026-05-23,2026-05-22,-1.0
9,FKO000401,2026-03-29,2026-03-29,0.0


In [61]:
# Calculate average delivery delay by city

city_delay_summary = orders_clean.groupby("customer_city")["delivery_delay_days"].mean().round(2)

print(city_delay_summary)

customer_city
Ahmedabad    1.90
Bengaluru    1.44
Chennai      1.74
Delhi        1.21
Hyderabad    1.60
Indore       1.89
Jaipur       1.81
Kolkata      1.98
Lucknow      1.08
Mumbai       1.75
Noida        1.30
Pune         1.85
Name: delivery_delay_days, dtype: float64


In [62]:
# Displaying top 5 cities by average delivery delay

top_5_cities = orders_clean.groupby("customer_city")[
    "delivery_delay_days"].mean().round(2).sort_values(ascending=False).head(5)

display(top_5_cities)

,delivery_delay_days
customer_city,
Kolkata,1.98
Ahmedabad,1.90
Indore,1.89
Pune,1.85
Jaipur,1.81


## Milestone 6 · Connect refunds to original orders
**From the team:** Finance can investigate refund requests, but needs the corresponding order context. Some orders have no refund request; those orders must remain in the overview.

**Your responsibility:** Investigate refund outcomes, including requests pending or failed. Summarize the refund evidence *at order level* before joining to orders, keeping all original orders in the result. Contrast requested refund amounts and identify high-value cases.

**Handover evidence:** Create **`refunds_by_order`** with one row per `order_id` and the fields `refund_count`, `total_refund_amount`, `pending_or_failed_refund_count`. Create **`orders_refunds_df`** with one row per order. Show refund-status and high-value case summaries.

In [63]:
# Milestone 06: Connect refunds to original orders
# Implement your approach below and display the requested evidence.

In [64]:
refunds_clean.head()

,refund_id,order_id,refund_amount,refund_date,refund_status
0,RFD000291,FKO000086,15535,2026-06-16,Completed
1,RFD000103,FKO000927,4051,2026-07-01,Completed
2,RFD000518,FKO000907,8761,2026-02-24,Completed
3,RFD000217,FKO000233,16436,2026-04-12,Failed
4,RFD000800,FKO000474,2047,2026-05-02,Completed


In [65]:
# Milestone 6: Count refunds per order

refund_summary = refunds_clean.groupby("order_id").size().reset_index(name="refund_count")

display(refund_summary.head())

,order_id,refund_count
0,FKO000002,2
1,FKO000003,1
2,FKO000004,1
3,FKO000005,2
4,FKO000008,1


In [66]:
# Calculate total refund amount per order

refund_amount_summary = refunds_clean.groupby(
    "order_id"
)["refund_amount"].sum().reset_index(name="total_refund_amount")

display(refund_amount_summary.head())

,order_id,total_refund_amount
0,FKO000002,41140
1,FKO000003,9213
2,FKO000004,14785
3,FKO000005,15427
4,FKO000008,6818


In [67]:
# Count pending or failed refunds per order

pending_failed = refunds_clean[refunds_clean["refund_status"].isin(["Pending", "Failed"])]

pending_failed_summary = pending_failed.groupby(
    "order_id"
).size().reset_index(name="pending_or_failed_refund_count")

display(pending_failed_summary.head())

,order_id,pending_or_failed_refund_count
0,FKO000005,1
1,FKO000011,1
2,FKO000012,1
3,FKO000015,1
4,FKO000016,2


In [68]:
# Combine refund summaries

refund_summary = refund_summary.merge(
    refund_amount_summary, on="order_id", how="outer"
)

refund_summary = refund_summary.merge(
    pending_failed_summary, on="order_id", how="outer"
)

display(refund_summary.head(10))

,order_id,refund_count,total_refund_amount,pending_or_failed_refund_count
0,FKO000002,2,41140,NaN
1,FKO000003,1,9213,NaN
2,FKO000004,1,14785,NaN
3,FKO000005,2,15427,1.0
4,FKO000008,1,6818,NaN
5,FKO000011,1,1350,1.0
6,FKO000012,1,19464,1.0
7,FKO000013,2,8077,NaN
8,FKO000014,1,978,NaN
9,FKO000015,1,16474,1.0


In [69]:
print(refund_summary.isnull().sum())

order_id                            0
refund_count                        0
total_refund_amount                 0
pending_or_failed_refund_count    384
dtype: int64


In [70]:
# Fill missing summary counts and amounts with zero

refund_summary = refund_summary.fillna(0)

print("Missing values:")
print(refund_summary.isnull().sum())

display(refund_summary.head(10))

Missing values:
order_id                          0
refund_count                      0
total_refund_amount               0
pending_or_failed_refund_count    0
dtype: int64


,order_id,refund_count,total_refund_amount,pending_or_failed_refund_count
0,FKO000002,2,41140,0.0
1,FKO000003,1,9213,0.0
2,FKO000004,1,14785,0.0
3,FKO000005,2,15427,1.0
4,FKO000008,1,6818,0.0
5,FKO000011,1,1350,1.0
6,FKO000012,1,19464,1.0
7,FKO000013,2,8077,0.0
8,FKO000014,1,978,0.0
9,FKO000015,1,16474,1.0


In [71]:
# Merge refund summary with orders

orders_with_refunds = orders_clean.merge(
    refund_summary,
    on="order_id",
    how="left"
)

print("Total orders:", len(orders_with_refunds))
display(orders_with_refunds.head())

Total orders: 1600


,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status,delivery_delay_days,refund_count,total_refund_amount,pending_or_failed_refund_count
0,FKO000702,CUS00237,Bengaluru,2026-05-26,2026-05-30,NaT,Grocery,36082,Cancelled,NaN,1.0,20946.0,0.0
1,FKO000636,CUS00689,Hyderabad,2026-04-19,2026-04-25,2026-04-24,Grocery,47326,Delivered,-1.0,NaN,NaN,NaN
2,FKO001570,CUS00222,Pune,2026-01-23,2026-01-30,NaT,Home & Kitchen,43254,Shipped,NaN,NaN,NaN,NaN
3,FKO001303,CUS00035,Lucknow,2026-02-12,2026-02-17,2026-02-18,Home & Kitchen,22215,Delivered,1.0,1.0,6583.0,0.0
4,FKO001208,CUS00619,Lucknow,2026-07-08,2026-07-11,2026-07-09,Home & Kitchen,2512,Delivered,-2.0,2.0,24718.0,1.0


In [72]:
# Fill missing refund values with zero

refund_columns = [
    "refund_count",
    "total_refund_amount",
    "pending_or_failed_refund_count"
]

orders_with_refunds[refund_columns] = (
    orders_with_refunds[refund_columns].fillna(0)
)

print("Missing refund summary values:")
print(orders_with_refunds[refund_columns].isnull().sum())

print("\nTotal orders:", len(orders_with_refunds))

Missing refund summary values:
refund_count                      0
total_refund_amount               0
pending_or_failed_refund_count    0
dtype: int64

Total orders: 1600


In [73]:
# Verify Milestone 6

print("Total orders:", len(orders_with_refunds))

print("\nColumns:")
print(orders_with_refunds.columns.tolist())

print("\nFirst five records:")
display(orders_with_refunds.head())

Total orders: 1600

Columns:
['order_id', 'customer_id', 'customer_city', 'order_date', 'promised_delivery_date', 'actual_delivery_date', 'product_category', 'order_amount', 'order_status', 'delivery_delay_days', 'refund_count', 'total_refund_amount', 'pending_or_failed_refund_count']

First five records:


,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status,delivery_delay_days,refund_count,total_refund_amount,pending_or_failed_refund_count
0,FKO000702,CUS00237,Bengaluru,2026-05-26,2026-05-30,NaT,Grocery,36082,Cancelled,NaN,1.0,20946.0,0.0
1,FKO000636,CUS00689,Hyderabad,2026-04-19,2026-04-25,2026-04-24,Grocery,47326,Delivered,-1.0,0.0,0.0,0.0
2,FKO001570,CUS00222,Pune,2026-01-23,2026-01-30,NaT,Home & Kitchen,43254,Shipped,NaN,0.0,0.0,0.0
3,FKO001303,CUS00035,Lucknow,2026-02-12,2026-02-17,2026-02-18,Home & Kitchen,22215,Delivered,1.0,1.0,6583.0,0.0
4,FKO001208,CUS00619,Lucknow,2026-07-08,2026-07-11,2026-07-09,Home & Kitchen,2512,Delivered,-2.0,2.0,24718.0,1.0


## Milestone 7 · Find repeat complaint patterns
**From the team:** Support management wants to know which issues are most common, which complaints are unresolved, and which customers or orders have repeat contact.

**Your responsibility:** Analyze complaint categories and resolution statuses. Investigate repeat complaints by order and repeat contacts by customer, without treating multiple distinct complaints as duplicate records.

**Handover evidence:** Create **`complaints_by_order`** with one row per `order_id` and fields `complaint_count`, `open_complaint_count` (open means status `Open`). Display repeated-order and repeated-customer cases and your leading complaint categories.

In [74]:
# Milestone 07: Find repeat complaint patterns
# Implement your approach below and display the requested evidence.

In [75]:
# Milestone 7: Count complaints per order

complaints_by_order = complaints_clean.groupby(
    "order_id"
).size().reset_index(name="complaint_count")

display(complaints_by_order.head())

,order_id,complaint_count
0,FKO000001,1
1,FKO000003,1
2,FKO000004,1
3,FKO000006,2
4,FKO000007,1


In [76]:
# Count open complaints per order

open_complaints = complaints_clean[complaints_clean["complaint_status"] == "Open"]

open_complaints_by_order = open_complaints.groupby(
    "order_id"
).size().reset_index(name="open_complaint_count")

display(open_complaints_by_order.head())

,order_id,open_complaint_count
0,FKO000001,1
1,FKO000006,1
2,FKO000007,1
3,FKO000014,1
4,FKO000019,1


In [77]:
# Combine complaint summaries

complaints_by_order = complaints_by_order.merge(
    open_complaints_by_order,
    on="order_id",
    how="left"
)

complaints_by_order["open_complaint_count"] = (
    complaints_by_order["open_complaint_count"].fillna(0)
)

complaints_by_order["open_complaint_count"] = (
    complaints_by_order["open_complaint_count"].astype(int)
)

display(complaints_by_order.head())

,order_id,complaint_count,open_complaint_count
0,FKO000001,1,1
1,FKO000003,1,0
2,FKO000004,1,0
3,FKO000006,2,1
4,FKO000007,1,1


In [78]:
# Find orders with repeat complaints

repeat_complaints = complaints_by_order[
    complaints_by_order["complaint_count"] > 1
]

print("Orders with repeat complaints:", len(repeat_complaints))

display(repeat_complaints.head(10))

Orders with repeat complaints: 406


,order_id,complaint_count,open_complaint_count
3,FKO000006,2,1
6,FKO000014,3,1
9,FKO000022,2,0
16,FKO000033,3,0
18,FKO000036,2,0
20,FKO000039,3,0
24,FKO000044,2,1
26,FKO000048,2,0
28,FKO000050,2,1
29,FKO000051,2,1


In [79]:
# Count complaints per customer

complaints_by_customer = complaints_clean.groupby(
    "customer_id"
).size().reset_index(name="complaint_count")

# Find customers with repeat complaints

repeat_customers = complaints_by_customer[
    complaints_by_customer["complaint_count"] > 1
]

print("Customers with repeat complaints:", len(repeat_customers))

display(repeat_customers.head(10))

Customers with repeat complaints: 376


,customer_id,complaint_count
0,CUS00001,2
2,CUS00004,2
4,CUS00007,3
5,CUS00012,8
6,CUS00013,2
7,CUS00014,5
8,CUS00015,4
10,CUS00018,6
11,CUS00019,2
12,CUS00020,3


In [80]:
# Find the most common complaint categories
print(complaints_clean.columns.tolist())

['complaint_id', 'order_id', 'customer_id', 'complaint_date', 'complaint_type', 'complaint_status']


In [81]:
# Find the top 5 complaint categories

top_complaint_categories = complaints_clean[
    "complaint_type"
].value_counts().head(5)

display(top_complaint_categories)

,count
complaint_type,
Delivery Delay,240
Product Quality,234
Missing Item,209
Cancellation Issue,203
Damaged Product,200


In [82]:
# Final check for complaints_by_order

print("Total rows:", len(complaints_by_order))
print("Unique order IDs:", complaints_by_order["order_id"].nunique())

print("\nColumns:")
print(complaints_by_order.columns.tolist())

display(complaints_by_order.head())

Total rows: 967
Unique order IDs: 967

Columns:
['order_id', 'complaint_count', 'open_complaint_count']


,order_id,complaint_count,open_complaint_count
0,FKO000001,1,1
1,FKO000003,1,0
2,FKO000004,1,0
3,FKO000006,2,1
4,FKO000007,1,1


In [83]:
# Display orders with repeat complaints

display(repeat_complaints.head(10))

,order_id,complaint_count,open_complaint_count
3,FKO000006,2,1
6,FKO000014,3,1
9,FKO000022,2,0
16,FKO000033,3,0
18,FKO000036,2,0
20,FKO000039,3,0
24,FKO000044,2,1
26,FKO000048,2,0
28,FKO000050,2,1
29,FKO000051,2,1


In [84]:
# Display customers with repeat complaints

display(repeat_customers.head(10))

,customer_id,complaint_count
0,CUS00001,2
2,CUS00004,2
4,CUS00007,3
5,CUS00012,8
6,CUS00013,2
7,CUS00014,5
8,CUS00015,4
10,CUS00018,6
11,CUS00019,2
12,CUS00020,3


### Milestone 7: Repeat Complaint Analysis

- Created `complaints_by_order` with one row per order containing complaints.
- Calculated `complaint_count` and `open_complaint_count` for each order.
- Identified orders with repeat complaints and customers with repeat contacts.
- Analyzed complaint categories to identify the most common issues.
- Found that **Delivery Delay** was the leading complaint category with 240 complaints, followed by Product Quality (234), Missing Item (209), Cancellation Issue (203), and Damaged Product (200).
- Preserved multiple distinct complaints instead of treating them as duplicate records.

## Milestone 8 · Deliver one order investigation view
**From the team:** Support agents should no longer need to manually look up the three source files for every case. The lead wants a dependable one-row-per-order working table.

**Your responsibility:** Combine cleaned order details with the aggregated refund and complaint information. Retain orders with no associated issues, represent absent issue counts as zero, and verify no orders were accidentally duplicated or lost when joining. Note any issue records referring to unknown orders.

**Handover evidence:** Create **`final_orders_df`** with one row per order, including `order_id`, `customer_id`, `customer_city`, `order_status`, `delivery_delay_days`, `refund_count`, `total_refund_amount`, `pending_or_failed_refund_count`, `complaint_count`, `open_complaint_count`. Show a row-count reconciliation.

In [85]:
# Milestone 08: Deliver one order investigation view
# Implement your approach below and display the requested evidence.

In [86]:
# Milestone 8: Start the final order investigation table

final_orders_df = orders_with_refunds.copy()

print("Orders before adding complaints:", len(final_orders_df))
display(final_orders_df.head())

Orders before adding complaints: 1600


,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status,delivery_delay_days,refund_count,total_refund_amount,pending_or_failed_refund_count
0,FKO000702,CUS00237,Bengaluru,2026-05-26,2026-05-30,NaT,Grocery,36082,Cancelled,NaN,1.0,20946.0,0.0
1,FKO000636,CUS00689,Hyderabad,2026-04-19,2026-04-25,2026-04-24,Grocery,47326,Delivered,-1.0,0.0,0.0,0.0
2,FKO001570,CUS00222,Pune,2026-01-23,2026-01-30,NaT,Home & Kitchen,43254,Shipped,NaN,0.0,0.0,0.0
3,FKO001303,CUS00035,Lucknow,2026-02-12,2026-02-17,2026-02-18,Home & Kitchen,22215,Delivered,1.0,1.0,6583.0,0.0
4,FKO001208,CUS00619,Lucknow,2026-07-08,2026-07-11,2026-07-09,Home & Kitchen,2512,Delivered,-2.0,2.0,24718.0,1.0


In [87]:
print(orders_with_refunds.columns.tolist())

['order_id', 'customer_id', 'customer_city', 'order_date', 'promised_delivery_date', 'actual_delivery_date', 'product_category', 'order_amount', 'order_status', 'delivery_delay_days', 'refund_count', 'total_refund_amount', 'pending_or_failed_refund_count']


In [88]:
print(final_orders_df.columns.tolist())

['order_id', 'customer_id', 'customer_city', 'order_date', 'promised_delivery_date', 'actual_delivery_date', 'product_category', 'order_amount', 'order_status', 'delivery_delay_days', 'refund_count', 'total_refund_amount', 'pending_or_failed_refund_count']


In [89]:
# Add complaint information to the final table

final_orders_df = final_orders_df.merge(
    complaints_by_order,
    on="order_id",
    how="left"
)

print("Orders after adding complaints:", len(final_orders_df))
display(final_orders_df.head())

Orders after adding complaints: 1600


,order_id,customer_id,customer_city,order_date,promised_delivery_date,actual_delivery_date,product_category,order_amount,order_status,delivery_delay_days,refund_count,total_refund_amount,pending_or_failed_refund_count,complaint_count,open_complaint_count
0,FKO000702,CUS00237,Bengaluru,2026-05-26,2026-05-30,NaT,Grocery,36082,Cancelled,NaN,1.0,20946.0,0.0,2.0,0.0
1,FKO000636,CUS00689,Hyderabad,2026-04-19,2026-04-25,2026-04-24,Grocery,47326,Delivered,-1.0,0.0,0.0,0.0,1.0,1.0
2,FKO001570,CUS00222,Pune,2026-01-23,2026-01-30,NaT,Home & Kitchen,43254,Shipped,NaN,0.0,0.0,0.0,NaN,NaN
3,FKO001303,CUS00035,Lucknow,2026-02-12,2026-02-17,2026-02-18,Home & Kitchen,22215,Delivered,1.0,1.0,6583.0,0.0,1.0,0.0
4,FKO001208,CUS00619,Lucknow,2026-07-08,2026-07-11,2026-07-09,Home & Kitchen,2512,Delivered,-2.0,2.0,24718.0,1.0,1.0,1.0


In [90]:
# Fill missing complaint counts with zero

final_orders_df["complaint_count"] = (
    final_orders_df["complaint_count"].fillna(0)
)

final_orders_df["open_complaint_count"] = (
    final_orders_df["open_complaint_count"].fillna(0)
)

print("Missing complaint counts:")
print(final_orders_df[
    ["complaint_count", "open_complaint_count"]
].isnull().sum())

Missing complaint counts:
complaint_count         0
open_complaint_count    0
dtype: int64


In [91]:
# Verify final order counts

print("Orders before merging:", len(orders_clean))
print("Orders in final table:", len(final_orders_df))

print("Unique order IDs:",final_orders_df["order_id"].nunique())
print("Duplicate order IDs:",final_orders_df["order_id"].duplicated().sum())

Orders before merging: 1600
Orders in final table: 1600
Unique order IDs: 1600
Duplicate order IDs: 0


In [92]:
# Select the required columns

required_columns = [
    "order_id",
    "customer_id",
    "customer_city",
    "order_status",
    "delivery_delay_days",
    "refund_count",
    "total_refund_amount",
    "pending_or_failed_refund_count",
    "complaint_count",
    "open_complaint_count"
]

final_orders_df = final_orders_df[required_columns]

print("Final dataset shape:", final_orders_df.shape)
print("\nColumn names:")
print(final_orders_df.columns.tolist())

display(final_orders_df.head())

Final dataset shape: (1600, 10)

Column names:
['order_id', 'customer_id', 'customer_city', 'order_status', 'delivery_delay_days', 'refund_count', 'total_refund_amount', 'pending_or_failed_refund_count', 'complaint_count', 'open_complaint_count']


,order_id,customer_id,customer_city,order_status,delivery_delay_days,refund_count,total_refund_amount,pending_or_failed_refund_count,complaint_count,open_complaint_count
0,FKO000702,CUS00237,Bengaluru,Cancelled,NaN,1.0,20946.0,0.0,2.0,0.0
1,FKO000636,CUS00689,Hyderabad,Delivered,-1.0,0.0,0.0,0.0,1.0,1.0
2,FKO001570,CUS00222,Pune,Shipped,NaN,0.0,0.0,0.0,0.0,0.0
3,FKO001303,CUS00035,Lucknow,Delivered,1.0,1.0,6583.0,0.0,1.0,0.0
4,FKO001208,CUS00619,Lucknow,Delivered,-2.0,2.0,24718.0,1.0,1.0,1.0


In [93]:
final_orders_df.dtypes

,0
order_id,object
customer_id,object
customer_city,object
order_status,object
delivery_delay_days,float64
refund_count,float64
total_refund_amount,float64
pending_or_failed_refund_count,float64
complaint_count,float64
open_complaint_count,float64


In [94]:
# Convert count columns to integers

count_columns = [
    "refund_count",
    "pending_or_failed_refund_count",
    "complaint_count",
    "open_complaint_count"
]

final_orders_df[count_columns] = (final_orders_df[count_columns].astype(int))

display(final_orders_df.head())

,order_id,customer_id,customer_city,order_status,delivery_delay_days,refund_count,total_refund_amount,pending_or_failed_refund_count,complaint_count,open_complaint_count
0,FKO000702,CUS00237,Bengaluru,Cancelled,NaN,1,20946.0,0,2,0
1,FKO000636,CUS00689,Hyderabad,Delivered,-1.0,0,0.0,0,1,1
2,FKO001570,CUS00222,Pune,Shipped,NaN,0,0.0,0,0,0
3,FKO001303,CUS00035,Lucknow,Delivered,1.0,1,6583.0,0,1,0
4,FKO001208,CUS00619,Lucknow,Delivered,-2.0,2,24718.0,1,1,1


In [95]:
final_orders_df.dtypes

,0
order_id,object
customer_id,object
customer_city,object
order_status,object
delivery_delay_days,float64
refund_count,int64
total_refund_amount,float64
pending_or_failed_refund_count,int64
complaint_count,int64
open_complaint_count,int64


In [96]:
# Final validation

print("Total rows:", len(final_orders_df))
print("Unique orders:", final_orders_df["order_id"].nunique())
print("Duplicate order IDs:",final_orders_df["order_id"].duplicated().sum())

print("\nMissing values by column:")
print(final_orders_df.isnull().sum())

Total rows: 1600
Unique orders: 1600
Duplicate order IDs: 0

Missing values by column:
order_id                            0
customer_id                         0
customer_city                       0
order_status                        0
delivery_delay_days               487
refund_count                        0
total_refund_amount                 0
pending_or_failed_refund_count      0
complaint_count                     0
open_complaint_count                0
dtype: int64


In [97]:
# Row-count reconciliation

print("Original cleaned orders:", len(orders_clean))
print("Final investigation table:", len(final_orders_df))
print("Difference:", len(orders_clean) - len(final_orders_df))
print("Unique final order IDs:", final_orders_df["order_id"].nunique())

Original cleaned orders: 1600
Final investigation table: 1600
Difference: 0
Unique final order IDs: 1600


### Milestone 8: Order Investigation View

- Created `final_orders_df` by combining cleaned order details with aggregated refund and complaint information.
- Retained all 1,600 orders, including orders without associated refunds or complaints.
- Verified that the final table contains 1,600 rows, 1,600 unique order IDs, and no duplicate order IDs.
- Filled missing refund and complaint summary counts and amounts with zero.
- Retained 487 missing delivery delays because the actual delivery dates were unavailable.
- Identified 18 refund records and 21 complaint records linked to unknown orders. These records were retained for investigation.

## Milestone 9 · Prepare a support triage queue
**From the team:** The Support Lead needs transparent rules for which customers to contact first. For this training case, priority reflects the number of important issue signals, not an opaque predictive model.

**Your responsibility:** Flag (a) delivered orders more than three days late, (b) orders with at least one pending or failed refund, and (c) orders with multiple complaints. Assign `High` when at least two signals apply, `Medium` for exactly one, and `Low` otherwise. Surface incomplete delivery-date records separately rather than assuming they were on time.

**Handover evidence:** Add **`support_priority`** to **`final_orders_df`** and retain the three traceable Boolean signal columns: `delayed_over_3_days`, `refund_needs_attention`, `repeated_complaints`. Produce a priority count and a top-ten High-priority case preview.

In [98]:
# Milestone 09: Prepare a support triage queue
# Implement your approach below and display the requested evidence.

In [99]:
# Milestone 9: Create support priority signals

final_orders_df["delayed_over_3_days"] = (
    (final_orders_df["order_status"] == "Delivered") &
    (final_orders_df["delivery_delay_days"] > 3)
)

final_orders_df["refund_needs_attention"] = (
    final_orders_df["pending_or_failed_refund_count"] > 0
)

final_orders_df["repeated_complaints"] = (
    final_orders_df["complaint_count"] > 1
)

display(final_orders_df[
    ["order_id", "order_status", "delivery_delay_days",
     "pending_or_failed_refund_count", "complaint_count",
     "delayed_over_3_days", "refund_needs_attention",
     "repeated_complaints"]].head(10))

,order_id,order_status,delivery_delay_days,pending_or_failed_refund_count,complaint_count,delayed_over_3_days,refund_needs_attention,repeated_complaints
0,FKO000702,Cancelled,NaN,0,2,False,False,True
1,FKO000636,Delivered,-1.0,0,1,False,False,False
2,FKO001570,Shipped,NaN,0,0,False,False,False
3,FKO001303,Delivered,1.0,0,1,False,False,False
4,FKO001208,Delivered,-2.0,1,1,False,True,False
5,FKO001595,Delivered,4.0,0,0,True,False,False
6,FKO000849,Pending,NaN,2,0,False,True,False
7,FKO001348,Pending,NaN,1,0,False,True,False
8,FKO001241,Delivered,-1.0,0,2,False,False,True
9,FKO000401,Delivered,0.0,0,0,False,False,False


In [100]:
# Count support issue signals

final_orders_df["issue_signal_count"] = (
    final_orders_df["delayed_over_3_days"].astype(int)
    + final_orders_df["refund_needs_attention"].astype(int)
    + final_orders_df["repeated_complaints"].astype(int)
)

display(final_orders_df[["order_id", "issue_signal_count"]].head(10))

,order_id,issue_signal_count
0,FKO000702,1
1,FKO000636,0
2,FKO001570,0
3,FKO001303,0
4,FKO001208,1
5,FKO001595,1
6,FKO000849,1
7,FKO001348,1
8,FKO001241,1
9,FKO000401,0


In [101]:
# Assign support priority

final_orders_df["support_priority"] = "Low"

final_orders_df.loc[
    final_orders_df["issue_signal_count"] == 1, "support_priority"
] = "Medium"

final_orders_df.loc[
    final_orders_df["issue_signal_count"] >= 2, "support_priority"
] = "High"

# Count orders in each priority level
print(final_orders_df["support_priority"].value_counts())

support_priority
Medium    682
Low       682
High      236
Name: count, dtype: int64


In [102]:
# Identify orders with incomplete delivery dates

incomplete_delivery_dates = final_orders_df[
    final_orders_df["delivery_delay_days"].isna()
]

print("Orders with incomplete delivery dates:", len(incomplete_delivery_dates))

display(incomplete_delivery_dates[
    ["order_id", "customer_id", "order_status",
     "delivery_delay_days", "support_priority"]
].head(10))

Orders with incomplete delivery dates: 487


,order_id,customer_id,order_status,delivery_delay_days,support_priority
0,FKO000702,CUS00237,Cancelled,NaN,Medium
2,FKO001570,CUS00222,Shipped,NaN,Low
6,FKO000849,CUS00120,Pending,NaN,Medium
7,FKO001348,CUS00108,Pending,NaN,Medium
13,FKO000126,CUS00100,Pending,NaN,Low
15,FKO000023,CUS00630,Shipped,NaN,Medium
18,FKO000409,CUS00754,Pending,NaN,Low
21,FKO000393,CUS00067,Shipped,NaN,Medium
23,FKO000006,CUS00671,Shipped,NaN,Medium
25,FKO001158,CUS00232,Shipped,NaN,Low


In [103]:
# Display the top 10 High-priority orders

high_priority_cases = final_orders_df[
    final_orders_df["support_priority"] == "High"
]

top_10_high_priority = high_priority_cases.sort_values(
    by="issue_signal_count",
    ascending=False
).head(10)

display(top_10_high_priority[
    ["order_id", "customer_id", "order_status",
     "delivery_delay_days", "refund_count",
     "pending_or_failed_refund_count", "complaint_count",
     "issue_signal_count", "support_priority"]
])

,order_id,customer_id,order_status,delivery_delay_days,refund_count,pending_or_failed_refund_count,complaint_count,issue_signal_count,support_priority
157,FKO000844,CUS00664,Delivered,4.0,2,1,2,3,High
102,FKO000586,CUS00190,Delivered,4.0,3,3,2,3,High
85,FKO001090,CUS00368,Delivered,10.0,2,1,2,3,High
84,FKO000212,CUS00015,Delivered,10.0,2,1,2,3,High
605,FKO000835,CUS00185,Delivered,4.0,3,2,2,3,High
576,FKO001193,CUS00040,Delivered,7.0,4,2,2,3,High
703,FKO000852,CUS00735,Delivered,4.0,2,1,2,3,High
430,FKO001182,CUS00721,Delivered,4.0,1,1,2,3,High
929,FKO001404,CUS00264,Delivered,10.0,2,1,2,3,High
854,FKO001420,CUS00273,Delivered,5.0,1,1,2,3,High


**Milestone 9: Support Triage Queue**

Created three issue signals to identify delayed deliveries, pending or failed refunds, and repeated complaints. Assigned High, Medium, and Low priorities based on the number of signals. Identified incomplete delivery-date records separately and generated priority counts and a preview of the top 10 High-priority cases.

## Milestone 10 · Hand over the operational report
**From the team:** Your final handover should let a support supervisor open the findings without running Python and understand the issue volume at a glance.

**Your responsibility:** Produce a useful detailed order-level report ordered High → Medium → Low, plus a short priority-level summary. Export the two outputs and explain three to five findings using actual values from your analysis.

**Handover evidence:** Create **`final_report`** and **`priority_summary`** DataFrames; `final_report` must contain `order_id` and `support_priority`, and `priority_summary` must contain `support_priority` and `order_count`. Export **`flipkart_order_issue_report.csv`** and **`flipkart_issue_summary.xlsx`** to **`DATA_DIR`**. Write findings beneath this task.

In [104]:
# Milestone 10: Hand over the operational report
# Implement your approach below and display the requested evidence.

In [105]:
# Milestone 10: Create the final order-level report

priority_order = ["High", "Medium", "Low"]

final_report = final_orders_df.sort_values(
    by="support_priority",
    key=lambda x: x.map({"High": 1, "Medium": 2, "Low": 3})
).copy()

display(final_report.head(10))
print("Total report rows:", len(final_report))

,order_id,customer_id,customer_city,order_status,delivery_delay_days,refund_count,total_refund_amount,pending_or_failed_refund_count,complaint_count,open_complaint_count,delayed_over_3_days,refund_needs_attention,repeated_complaints,issue_signal_count,support_priority
1597,FKO001239,CUS00213,Jaipur,Delivered,5.0,1,1815.0,1,1,0,True,True,False,2,High
40,FKO000259,CUS00536,Lucknow,Delivered,4.0,2,24223.0,0,2,0,True,False,True,2,High
41,FKO000660,CUS00418,Indore,Delivered,5.0,2,26808.0,1,0,0,True,True,False,2,High
43,FKO001146,CUS00660,Mumbai,Delivered,-1.0,1,18422.0,1,3,1,False,True,True,2,High
46,FKO001332,CUS00290,Lucknow,Delivered,4.0,1,8700.0,1,0,0,True,True,False,2,High
1552,FKO001113,CUS00550,Noida,Delivered,4.0,1,3620.0,1,1,1,True,True,False,2,High
1554,FKO000496,CUS00426,Hyderabad,Delivered,4.0,2,37089.0,1,0,0,True,True,False,2,High
1562,FKO000423,CUS00629,Chennai,Delivered,-1.0,1,20219.0,1,3,1,False,True,True,2,High
943,FKO000534,CUS00383,Jaipur,Delivered,10.0,1,11297.0,1,0,0,True,True,False,2,High
915,FKO000963,CUS00613,Kolkata,Delivered,5.0,0,0.0,0,2,0,True,False,True,2,High


Total report rows: 1600


In [106]:
# Create priority-level summary

priority_summary = (
    final_report.groupby("support_priority")
    .size()
    .reset_index(name="order_count")
)

priority_summary["support_priority"] = pd.Categorical(
    priority_summary["support_priority"],
    categories=["High", "Medium", "Low"],
    ordered=True
)

priority_summary = priority_summary.sort_values(
    "support_priority"
)

priority_summary["support_priority"] = (
    priority_summary["support_priority"].astype(str)
)

display(priority_summary)

,support_priority,order_count
0,High,236
2,Medium,682
1,Low,682


In [107]:
# Export the detailed order report

final_report.to_csv(
    f"{DATA_DIR}/flipkart_order_issue_report.csv",
    index=False
)

print("Detailed report exported successfully.")

Detailed report exported successfully.


In [108]:
# Export the priority summary to Excel

priority_summary.to_excel(
    f"{DATA_DIR}/flipkart_issue_summary.xlsx",
    index=False
)

print("Priority summary exported successfully.")

Priority summary exported successfully.


In [109]:
# Verify the exported files

import os

print("CSV report exists:",
      os.path.exists(f"{DATA_DIR}/flipkart_order_issue_report.csv"))

print("Excel summary exists:",
      os.path.exists(f"{DATA_DIR}/flipkart_issue_summary.xlsx"))

CSV report exists: True
Excel summary exists: True


### Key Findings

1. **Order volume:** The final report contains 1,600 unique orders.
2. **Support priority:** 236 orders are High priority, 682 are Medium priority, and 682 are Low priority.
3. **Delivery delays:** 234 orders were delivered more than three days late.
4. **Refund issues:** 540 orders have at least one pending or failed refund requiring attention.
5. **Repeated complaints:** 405 orders have multiple complaints, indicating cases that may need additional support follow-up.

**Milestone 10: Operational Report Handover**

Created the detailed order-level report and priority summary. Exported both reports as CSV and Excel files to `DATA_DIR`. Documented five key findings covering order volume, support priorities, delivery delays, refund issues, and repeated complaints.

### Handover notes · What should the Support Lead know?
Write **3–5 observations** based on numbers in your actual outputs. Include one operational implication and at least one data-quality limitation.

**Your findings:**

- The analysis covers **1,600 unique orders**: 236 High priority, 682 Medium priority, and 682 Low priority.
- **236 orders (14.75%)** are High priority and should be reviewed first by the support team.
- **234 orders** were delivered more than three days late, highlighting delivery delays that may require operational follow-up.
- **540 orders** have pending or failed refunds, making refund resolution an important support task. A single order may have multiple issue signals, so these counts can overlap.
- **Data-quality limitation:** 487 orders have missing actual delivery dates, so their delivery delays cannot be calculated reliably. These records were kept separate for review rather than assumed to be on time.

In [110]:
print("City delay summary type:", type(city_delay_summary))
print(city_delay_summary.head())

City delay summary type: <class 'pandas.core.series.Series'>
customer_city
Ahmedabad    1.90
Bengaluru    1.44
Chennai      1.74
Delhi        1.21
Hyderabad    1.60
Name: delivery_delay_days, dtype: float64


In [111]:
# Create a city-level delayed order count summary

city_delay_summary = (
    orders_clean[
        orders_clean["delivery_delay_days"] > 0
    ]
    .groupby("customer_city")
    .size()
    .reset_index(name="delayed_order_count")
)

display(city_delay_summary.head())

,customer_city,delayed_order_count
0,Ahmedabad,59
1,Bengaluru,53
2,Chennai,58
3,Delhi,46
4,Hyderabad,50


In [112]:
# Match earlier milestone outputs to assessment names

refunds_by_order = refund_summary.copy()
orders_refunds_df = orders_with_refunds.copy()

In [113]:
print("Refund summary columns:")
print(refunds_by_order.columns.tolist())

print("\nRefund summary has unique order IDs:",
      refunds_by_order["order_id"].is_unique)

print("\nOrders-refunds table rows:",
      len(orders_refunds_df))

Refund summary columns:
['order_id', 'refund_count', 'total_refund_amount', 'pending_or_failed_refund_count']

Refund summary has unique order IDs: True

Orders-refunds table rows: 1600


In [114]:
print("Final report rows:", len(final_report))
print("Final report columns:", final_report.columns.tolist())

print("\nPriority summary:")
display(priority_summary)

print("\nCSV file exists:",
      (DATA_DIR / "flipkart_order_issue_report.csv").is_file())

print("Excel file exists:",
      (DATA_DIR / "flipkart_issue_summary.xlsx").is_file())

Final report rows: 1600
Final report columns: ['order_id', 'customer_id', 'customer_city', 'order_status', 'delivery_delay_days', 'refund_count', 'total_refund_amount', 'pending_or_failed_refund_count', 'complaint_count', 'open_complaint_count', 'delayed_over_3_days', 'refund_needs_attention', 'repeated_complaints', 'issue_signal_count', 'support_priority']

Priority summary:


,support_priority,order_count
0,High,236
2,Medium,682
1,Low,682



CSV file exists: True
Excel file exists: True


In [115]:
print("Refund ID missing:", refunds_clean["refund_id"].isna().sum())
print("Duplicate refund IDs:", refunds_clean["refund_id"].duplicated().sum())
print("Refund date type:", refunds_clean["refund_date"].dtype)
print("Refund status examples:")
print(refunds_clean["refund_status"].dropna().unique()[:10])

Refund ID missing: 0
Duplicate refund IDs: 21
Refund date type: datetime64[ns]
Refund status examples:
<StringArray>
['Completed', 'Failed', 'Pending']
Length: 3, dtype: string


In [116]:
print("Complaint ID missing:", complaints_clean["complaint_id"].isna().sum())
print("Duplicate complaint IDs:", complaints_clean["complaint_id"].duplicated().sum())
print("Complaint date type:", complaints_clean["complaint_date"].dtype)
print("Complaint status examples:")
print(complaints_clean["complaint_status"].dropna().unique()[:10])

Complaint ID missing: 0
Duplicate complaint IDs: 25
Complaint date type: datetime64[ns]
Complaint status examples:
['Resolved' 'Open' 'In Progress']


In [117]:
# Check exact duplicate refund rows

print(
    "Exact duplicate refund rows:",
    refunds_clean.duplicated().sum()
)

Exact duplicate refund rows: 21


In [118]:
# Remove exact duplicate rows from the cleaned refund data

refunds_clean = refunds_clean.drop_duplicates().copy()

print("Refund records after cleaning:", len(refunds_clean))
print("Remaining exact duplicates:", refunds_clean.duplicated().sum())
print("Remaining duplicate refund IDs:",
      refunds_clean["refund_id"].duplicated().sum())

Refund records after cleaning: 1350
Remaining exact duplicates: 0
Remaining duplicate refund IDs: 0


In [119]:
# Check exact duplicate complaint rows

print(
    "Exact duplicate complaint rows:",
    complaints_clean.duplicated().sum()
)

Exact duplicate complaint rows: 25


In [120]:
# Remove exact duplicate rows from cleaned complaint data

complaints_clean = complaints_clean.drop_duplicates().copy()

print("Complaint records after cleaning:", len(complaints_clean))
print("Remaining exact duplicates:", complaints_clean.duplicated().sum())
print("Remaining duplicate complaint IDs:",
      complaints_clean["complaint_id"].duplicated().sum())

Complaint records after cleaning: 1500
Remaining exact duplicates: 0
Remaining duplicate complaint IDs: 0


In [121]:
# Refresh complaint counts using cleaned complaint data

complaints_by_order = (
    complaints_clean.groupby("order_id")
    .size()
    .reset_index(name="complaint_count")
)

open_complaints = complaints_clean[
    complaints_clean["complaint_status"] == "Open"
]

open_complaints_by_order = (
    open_complaints.groupby("order_id")
    .size()
    .reset_index(name="open_complaint_count")
)

complaints_by_order = complaints_by_order.merge(
    open_complaints_by_order,
    on="order_id",
    how="left"
)

complaints_by_order["open_complaint_count"] = (
    complaints_by_order["open_complaint_count"].fillna(0).astype(int)
)

print("Cleaned complaint records:", len(complaints_clean))
print("Total complaints in summary:", complaints_by_order["complaint_count"].sum())
print("Unique orders in summary:", complaints_by_order["order_id"].nunique())

Cleaned complaint records: 1500
Total complaints in summary: 1500
Unique orders in summary: 967


In [122]:
# Refresh complaint counts in the final order table

final_orders_df = final_orders_df.drop(
    columns=["complaint_count", "open_complaint_count"]
)

final_orders_df = final_orders_df.merge(
    complaints_by_order,
    on="order_id",
    how="left"
)

final_orders_df["complaint_count"] = (
    final_orders_df["complaint_count"].fillna(0).astype(int)
)

final_orders_df["open_complaint_count"] = (
    final_orders_df["open_complaint_count"].fillna(0).astype(int)
)

print("Total orders:", len(final_orders_df))
print("Unique order IDs:", final_orders_df["order_id"].nunique())
print("Total complaints in final table:", final_orders_df["complaint_count"].sum())

Total orders: 1600
Unique order IDs: 1600
Total complaints in final table: 1480


In [123]:
# Refresh support signals after complaint cleanup

final_orders_df["repeated_complaints"] = (
    final_orders_df["complaint_count"] > 1
)

final_orders_df["issue_signal_count"] = (
    final_orders_df["delayed_over_3_days"].astype(int)
    + final_orders_df["refund_needs_attention"].astype(int)
    + final_orders_df["repeated_complaints"].astype(int)
)

final_orders_df["support_priority"] = "Low"

final_orders_df.loc[
    final_orders_df["issue_signal_count"] == 1,
    "support_priority"
] = "Medium"

final_orders_df.loc[
    final_orders_df["issue_signal_count"] >= 2,
    "support_priority"
] = "High"

print("Updated priority counts:")
print(final_orders_df["support_priority"].value_counts())

Updated priority counts:
support_priority
Low       688
Medium    678
High      234
Name: count, dtype: int64


In [124]:
# Refresh the final report in High → Medium → Low order

final_report = final_orders_df.sort_values(
    by="support_priority",
    key=lambda x: x.map({"High": 1, "Medium": 2, "Low": 3})
).copy()

# Refresh the priority summary
priority_summary = (
    final_report.groupby("support_priority")
    .size()
    .reindex(["High", "Medium", "Low"], fill_value=0)
    .rename_axis("support_priority")
    .reset_index(name="order_count")
)

# Export both updated files
final_report.to_csv(
    f"{DATA_DIR}/flipkart_order_issue_report.csv",
    index=False
)

priority_summary.to_excel(
    f"{DATA_DIR}/flipkart_issue_summary.xlsx",
    index=False
)

print("Final report and priority summary refreshed.")
print("Report rows:", len(final_report))
display(priority_summary)

Final report and priority summary refreshed.
Report rows: 1600


,support_priority,order_count
0,High,234
1,Medium,678
2,Low,688


In [125]:
# Refresh refund and complaint counts

refund_counts = refunds_clean.groupby("order_id").size()
complaint_counts = complaints_clean.groupby("order_id").size()

final_orders_df["refund_count"] = (
    final_orders_df["order_id"].map(refund_counts).fillna(0).astype(int)
)

final_orders_df["complaint_count"] = (
    final_orders_df["order_id"].map(complaint_counts).fillna(0).astype(int)
)

print("Refund and complaint counts refreshed.")

Refund and complaint counts refreshed.


In [126]:
# Final reconciliation check

for source, column in [
    (refunds_clean, "refund_count"),
    (complaints_clean, "complaint_count")
]:
    expected = (
        source[source["order_id"].isin(final_orders_df["order_id"])]
        .groupby("order_id")
        .size()
    )

    actual = final_orders_df.set_index("order_id")[column]
    expected = expected.reindex(actual.index, fill_value=0)

    print(column, "matches:", actual.equals(expected))

print("Total orders:", len(final_orders_df))
print("Unique order IDs:", final_orders_df["order_id"].nunique())

refund_count matches: True
complaint_count matches: True
Total orders: 1600
Unique order IDs: 1600


## Final milestone · Automated assessment
**Important:** Colab retains previously created variables and files. After changing or deleting any milestone code, choose **Runtime → Restart session**, then **Run all** before trusting the self-check. Running only the assessment cell can show incorrect passes. For official grading, the instructor must execute your saved `.ipynb` in a fresh kernel with clean input files.

Run the self-check **after all ten milestones**. It creates a scorecard for your notebook using observable DataFrame outputs and files. It does **not** require an answer key to be shared with students.

**How to read results:** `Pass` means the available output meets the named structural or business-rule check; `Review` means something is absent or differs from the expected contract. This is **formative self-checking**, not a secure examination: notebook code can be modified and passing checks do not certify every analysis table or commentary. For official grading, retain a human review of findings, data quality decisions, and a sample of calculations.

**Student instruction:** Run all cells from the top before evaluating. Do not edit the assessment cell.

In [127]:
# AUTOMATED MILESTONE ASSESSMENT — Run only after finishing Milestones 1–10.
# This is a non-secure learning aid; see the project brief for review expectations.
import pandas as pd
from pathlib import Path
from IPython.display import display

_scope = globals()
_checks = []
def check(task, criterion, fn):
    try:
        passed = bool(fn())
        msg = 'Meets check' if passed else 'Missing or inconsistent output'
    except Exception as ex:
        passed = False
        msg = f'Cannot verify: {type(ex).__name__}'
    _checks.append({'Milestone': task, 'Check': criterion, 'Result': 'Pass' if passed else 'Review', 'Points': 5 if passed else 0, 'Feedback': msg})

def df(name):
    obj = _scope.get(name)
    if not isinstance(obj, pd.DataFrame):
        raise ValueError(f'{name} is not a DataFrame')
    return obj

def has_cols(name, cols):
    return set(cols).issubset(df(name).columns)

def unique_nonnull(frame, key):
    return key in frame and frame[key].notna().all() and frame[key].is_unique

def numeric(frame, col):
    return col in frame and pd.api.types.is_numeric_dtype(frame[col])

def real_dates(frame, cols):
    return all(col in frame and pd.api.types.is_datetime64_any_dtype(frame[col]) for col in cols)

def normalized(frame, col):
    vals = frame[col].dropna().astype(str)
    return not vals.empty and vals.eq(vals.str.strip()).all() and vals.eq(vals.str.title()).all()

def expected_counts(name, key, source, id_col):
    return df(name)[key].sum() == len(df(source)) and df(name)[id_col].is_unique

def issues_match():
    o = df('final_orders_df')
    r = df('refunds_clean')
    c = df('complaints_clean')
    for src, col in [(r, 'refund_count'), (c, 'complaint_count')]:
        counts = src[src.order_id.isin(o.order_id)].groupby('order_id').size()
        actual = o.set_index('order_id')[col].fillna(0)
        aligned = counts.reindex(actual.index, fill_value=0)
        if not actual.eq(aligned).all(): return False
    return True

def priorities_match():
    o = df('final_orders_df')
    flags = ['delayed_over_3_days','refund_needs_attention','repeated_complaints']
    if not has_cols('final_orders_df', flags + ['support_priority']): return False
    if any(not pd.api.types.is_bool_dtype(o[x]) for x in flags): return False
    signals = o[flags].sum(axis=1)
    expected = signals.map(lambda x: 'High' if x >= 2 else ('Medium' if x == 1 else 'Low'))
    return o.support_priority.eq(expected).all()

def flags_match():
    o = df('final_orders_df')
    required = ['delayed_over_3_days','refund_needs_attention','repeated_complaints']
    if not set(required).issubset(o): return False
    delivered = o.order_status.astype('string').str.strip().str.casefold().eq('delivered')
    delay_expected = (delivered & o.delivery_delay_days.gt(3)).fillna(False)
    refund_expected = o.pending_or_failed_refund_count.fillna(0).gt(0)
    repeat_expected = o.complaint_count.fillna(0).gt(1)
    return all(o[col].fillna(False).eq(expected).all() for col, expected in zip(required,[delay_expected,refund_expected,repeat_expected]))

def report_matches():
    r = df('final_report')
    o = df('final_orders_df')
    return (has_cols('final_report',['order_id','support_priority']) and
            len(r)==len(o) and r.order_id.is_unique and
            set(r.order_id)==set(o.order_id) and
            r.set_index('order_id').support_priority.eq(o.set_index('order_id').loc[r.order_id,'support_priority'].to_numpy()).all())

def summary_matches():
    s = df('priority_summary')
    if not has_cols('priority_summary',['support_priority','order_count']): return False
    expected = df('final_orders_df').support_priority.value_counts()
    got = s.groupby('support_priority').order_count.sum()
    return got.reindex(expected.index,fill_value=0).eq(expected).all() and got.sum()==len(df('final_orders_df'))

check(1,'Orders loaded, key fields present',lambda: has_cols('orders_df',['order_id','customer_id','order_status','order_amount']))
check(1,'Orders source is non-empty',lambda: len(df('orders_df'))>0)
check(2,'Refund source loaded',lambda: has_cols('refunds_df',['refund_id','order_id','refund_status','refund_amount']) and len(df('refunds_df'))>0)
check(2,'Complaint source loaded',lambda: has_cols('complaints_df',['complaint_id','order_id','complaint_type','complaint_status']) and len(df('complaints_df'))>0)
check(3,'One clean record per order',lambda: unique_nonnull(df('orders_clean'),'order_id') and len(df('orders_clean'))<=len(df('orders_df')))
check(3,'Amounts, dates, statuses standardized',lambda: numeric(df('orders_clean'),'order_amount') and real_dates(df('orders_clean'),['order_date','promised_delivery_date','actual_delivery_date']) and normalized(df('orders_clean'),'order_status'))
check(4,'Refund records cleaned',lambda: unique_nonnull(df('refunds_clean'),'refund_id') and numeric(df('refunds_clean'),'refund_amount') and real_dates(df('refunds_clean'),['refund_date']) and normalized(df('refunds_clean'),'refund_status'))
check(4,'Complaint records cleaned',lambda: unique_nonnull(df('complaints_clean'),'complaint_id') and real_dates(df('complaints_clean'),['complaint_date']) and normalized(df('complaints_clean'),'complaint_status'))
check(5,'Delivery delay calculation is consistent',lambda: numeric(df('orders_clean'),'delivery_delay_days') and ((df('orders_clean').loc[df('orders_clean').order_status.eq('Delivered') & df('orders_clean').actual_delivery_date.notna() & df('orders_clean').promised_delivery_date.notna(),'actual_delivery_date'] - df('orders_clean').loc[df('orders_clean').order_status.eq('Delivered') & df('orders_clean').actual_delivery_date.notna() & df('orders_clean').promised_delivery_date.notna(),'promised_delivery_date']).dt.days.eq(df('orders_clean').loc[df('orders_clean').order_status.eq('Delivered') & df('orders_clean').actual_delivery_date.notna() & df('orders_clean').promised_delivery_date.notna(),'delivery_delay_days']).all()))
check(5,'City delay summary is available',lambda: has_cols('city_delay_summary',['customer_city','delayed_order_count']) and numeric(df('city_delay_summary'),'delayed_order_count'))
check(6,'Refund summary uses unique order IDs',lambda: has_cols('refunds_by_order',['order_id','refund_count','total_refund_amount','pending_or_failed_refund_count']) and df('refunds_by_order').order_id.is_unique)
check(6,'Refund join keeps order population',lambda: unique_nonnull(df('orders_refunds_df'),'order_id') and set(df('orders_refunds_df').order_id)==set(df('orders_clean').order_id))
check(7,'Complaints summarized by order',lambda: has_cols('complaints_by_order',['order_id','complaint_count','open_complaint_count']) and df('complaints_by_order').order_id.is_unique)
check(7,'Complaint totals match cleaned source',lambda: df('complaints_by_order').complaint_count.sum()==df('complaints_clean').order_id.notna().sum())
check(8,'Master table contains all orders once',lambda: unique_nonnull(df('final_orders_df'),'order_id') and set(df('final_orders_df').order_id)==set(df('orders_clean').order_id))
check(8,'Master issue counts reconcile',issues_match)
check(9,'Support signals follow business rules',flags_match)
check(9,'Priority mapping matches signals',priorities_match)
check(10,'Detailed report and export exist',lambda: report_matches() and (DATA_DIR/'flipkart_order_issue_report.csv').is_file())
check(10,'Priority summary and export exist',lambda: summary_matches() and (DATA_DIR/'flipkart_issue_summary.xlsx').is_file())

assessment_df = pd.DataFrame(_checks)
score_by_task = assessment_df.groupby('Milestone',as_index=False).agg(Points=('Points','sum'),Passed=('Result',lambda x:(x=='Pass').sum()))
score_by_task['Maximum'] = 10
score_by_task['Status'] = score_by_task['Passed'].map({0:'Review',1:'Partially verified',2:'Checks passed'})
print('AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE')
display(score_by_task[['Milestone','Points','Maximum','Status']])
print(f'Automated checks score: {assessment_df.Points.sum()}/100')
print('Checks needing attention:')
display(assessment_df.loc[assessment_df.Result=='Review',['Milestone','Check','Feedback']].reset_index(drop=True))
# Machine-readable output for independent, fresh-kernel instructor grading.
import json as _grade_json
print('FLIPKART_ASSESSMENT_JSON=' + _grade_json.dumps({'points': int(assessment_df.Points.sum()), 'milestones': score_by_task[['Milestone','Points','Maximum','Status']].to_dict('records'), 'checks': assessment_df[['Milestone','Check','Result','Points']].to_dict('records')}))


AUTOMATED SELF-ASSESSMENT · NOT FINAL HUMAN GRADE


,Milestone,Points,Maximum,Status
0,1,10,10,Checks passed
1,2,10,10,Checks passed
2,3,10,10,Checks passed
3,4,10,10,Checks passed
4,5,10,10,Checks passed
5,6,10,10,Checks passed
6,7,10,10,Checks passed
7,8,10,10,Checks passed
8,9,10,10,Checks passed
9,10,10,10,Checks passed


Automated checks score: 100/100
Checks needing attention:


,Milestone,Check,Feedback


FLIPKART_ASSESSMENT_JSON={"points": 100, "milestones": [{"Milestone": 1, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 2, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 3, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 4, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 5, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 6, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 7, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 8, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 9, "Points": 10, "Maximum": 10, "Status": "Checks passed"}, {"Milestone": 10, "Points": 10, "Maximum": 10, "Status": "Checks passed"}], "checks": [{"Milestone": 1, "Check": "Orders loaded, key fields present", "Result": "Pass", "Points": 5}, {"Milestone": 1, "Check": "Orders source is non-empty", "Result": "Pass", "Points": 5}, {"Milest

## Project Findings

1. **Order baseline:** The cleaned dataset contains 1,600 unique orders.
2. **Delivery performance:** Delivery delays were calculated by comparing actual and promised delivery dates. Missing actual delivery dates were retained rather than filled with assumed values.
3. **Refund analysis:** Refund records were cleaned and summarized by order, including refund counts, total refund amounts, and pending or failed refunds.
4. **Customer complaints:** Complaint records were cleaned and summarized by order to identify complaint frequency and open complaints.
5. **Support prioritization:** Orders were classified as High, Medium, or Low priority based on delivery delays, pending or failed refunds, and repeated complaints.
6. **Final deliverables:** The project produces a detailed order issue report in CSV format and a priority summary in Excel format.

**Conclusion:** This project combines order, refund, and customer complaint data into a single investigation view to help identify customer issues and prioritize support actions.

### Submission handover
1. Save your completed Colab copy in Drive and share its link with the required permissions, or submit the `.ipynb` file if instructed by your instructor.
2. Include the final automated scorecard **and** your written findings in the notebook.
3. Do not submit the original three input datasets unless requested; output CSV and Excel files are generated during your run.
